In [1]:
import pandas as pd
import numpy as np
import re 
from xgboost import XGBRanker
from sklearn.metrics import ndcg_score

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV 
from sklearn.metrics import classification_report, confusion_matrix, f1_score, hamming_loss, jaccard_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from scipy.stats import entropy

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
def clean_generic_name(name):
    if pd.isna(name) or name == 'nan':
        return name
    
    # 1. 統一標點符號：全型斜線與逗號轉半型，移除常見括號內容
    name = name.replace('／', '/').replace('，', ',')
    name = re.sub(r'[\(\uff08].*?[\)\uff09]', '', name)
    
    # 2. 強力移除劑量與單位 (處理 100,000, 1%, 80 mg/ml, 2.4 MIU 等)
    # 匹配數字、逗號、點、百分比，後接常見容量單位
    name = re.sub(r'[\d\.,\s]+(g|mg|ml|units|miu|％|%)\b', ' ', name, flags=re.IGNORECASE)
    # 針對單獨殘留的 /ml 或 %
    name = re.sub(r'/\s*ml|%', '', name, flags=re.IGNORECASE)
    
    # 3. 移除特定劑型關鍵字與無義詞
    forms = ['inj', 'tab', 'cap', 'susp', 'iv', 'oral', 'for', 'solution', 'hydrate', 'sodium', 'benzathine']
    pattern_forms = r'\b(' + '|'.join(forms) + r')\b'
    name = re.sub(pattern_forms, '', name, flags=re.IGNORECASE)

    # 4. 特定藥名結構處理 (依據您的要求)
    # Amphotericin B liposome -> Amphotericin B/liposome
    name = re.sub(r'Amphotericin B liposome', 'Amphotericin B/liposome', name, flags=re.IGNORECASE)

    # 5. 【核心強化】消除斜線 (/) 前後的任何空格
    # \s* 代表 0 到多個空白字元
    name = re.sub(r'\s*/\s*', '/', name)

    
    # 5. 清理殘留符號：移除多餘空格、末尾點號與斜線
    name = re.sub(r'\s+', ' ', name) # 多空格轉單空格
    name = name.strip(' ./,')       # 移除前後的空格、點、斜線、逗號
    
    # 6. 字典對照 (處理特殊轉換)
    mapping = {
        'R +I': 'ifampin/Isoniazid',
        'R 300 +I 150': 'ifampin/Isoniazid',
        'Penicillin G .': 'Penicillin G',
        'Penicillin G benzathine': 'Penicillin G',
        'Baktar': 'Sulfamethoxazole/Trimethoprim',
        'Clindamycin 1 ％' : 'Clindamycin',
        'Penicillin 5 MU' : 'Penicillin',
        'Rifampin, Isoniazid and Ethambutol' : 'Rifampin/Isoniazid/Ethambutol',
        'Minocycline injection' : 'Minocycline',
        'Amoxicillin/Clavulanate': 'Amoxicillin/Clavulanic acid'
    }
    
    # 如果完全符合字典 key，或是處理後變成 key 的樣子就轉換
    return mapping.get(name, name)

In [4]:
# 先讀取前兩列看看欄位名是什麼
temp_df = pd.read_csv(r'D:\ML Data\Spesis\table14.csv', nrows=2)
col_21 = temp_df.columns[21]
col_22 = temp_df.columns[22]
print(f"第 21 欄是: {col_21}, 第 22 欄是: {col_22}")

# 重新讀取並指定型態
table14 = pd.read_csv(r'D:\ML Data\Spesis\table14.csv', 
                      index_col=False, 
                      dtype={col_21: str, col_22: str})


第 21 欄是: CREATININE, 第 22 欄是: GPT


In [5]:
last_index = table14.groupby('ACCOUNTNO')['VERIFYDATE'].min().reset_index()
table14_last = pd.merge(table14, last_index, on=['ACCOUNTNO', 'VERIFYDATE'])

In [6]:
table14_last['drug'] = table14['GENERICNAME'].apply(clean_generic_name)

In [7]:
rank_mapping = {
    'A11': 1,
    'A21': 2,
    'A22': 2,
    'A32': 2,
    'A42': 3,
    'A52': 3,
    'C42': 3
}

In [8]:
table14_last['AUTIBIOTIC_GROUP'] = table14_last['AUTIBIOTICRANK'].map(rank_mapping)

In [9]:
table14_last = table14_last[['drug', 'AUTIBIOTIC_GROUP']] # GENERICNAME_Clear and AUTIBIOTICRANK

In [10]:
table14_clear = pd.read_csv(r'D:\ML Data\Spesis\table14_clear.csv', index_col=False)

In [11]:
table14_clear

,ACCOUNTNO,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Amphotericin B,Amphotericin B/liposome,Ampicillin,Ampicillin/Sulbactam,Azithromycin,Baloxavir marboxil,Cefadroxil,Cefazolin,Cefepime,Cefixime,Cefoperazone/sulbactam,Cefotaxime,Cefoxitin,Ceftazidime,Ceftazidime/Avibactam,Ceftizoxime,Ceftriaxone,Cefuroxime,Cephalexin,Ciprofloxacin,Clarithromycin,Clindamycin,Dicloxacillin,Doripenem,Doxycycline,Ertapenem,Erythromycin,Ethambutol,Famciclovir,Fenticonazole,Flomoxef,Fluconazole,Flucytosine,Fosfomycin,Ganciclovir,Gentamicin,Griseofulvin,Imipenem/Cilastatin,Isoniazid,Itraconazole,Lamivudine/Dolutegravir,Levofloxacin,Linezolid,Meropenem,Metronidazole,Micafungin,Minocycline,Moxifloxacin,Nemonoxacin,Nystatin,Oseltamivir,Oxacillin,Penicillin,Peramivir,Pipemidic acid,Piperacillin/Tazobactam,Pyrazinamide,Rifampin,Rifampin/Isoniazid/Ethambutol,Sulfamethoxazole/Trimethoprim,Teicoplanin,Telbivudine,Tenofovir,Tenofovir alafenamide,Terbinafine,Tetracycline,Valaciclovir,Vancomycin,Zanamivir,abacavir/lamivudine/dolutegravir,ifampin/Isoniazid,tenofovir/emtricitabine,tenofovir/emtricitabine/bictegravir,tenofovir/emtricitabine/rilpivirine,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag
0,I11300000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0,0,0
1,I11300000003,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
2,I11300000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
3,I11300000011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
4,I11300000014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27963,I11400060686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0,0,0
27964,I11400060687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0

In [12]:
start_index = table14_clear.columns.get_loc('Acyclovir')
end_index = table14_clear.columns.get_loc('tenofovir/emtricitabine/rilpivirine')

In [13]:
drug_cols = table14_clear.columns[start_index:end_index+1]
col_sum = table14_clear[drug_cols].sum()

In [14]:
final_cols = col_sum[col_sum >= 500].index.tolist()
base_cols = [c for c in table14_clear.columns if c not in drug_cols]
data_filter = table14_clear[base_cols + final_cols]

In [15]:
data_filter[final_cols].sum().sort_values(ascending=False)

Amoxicillin/Clavulanic acid    5515.0
Flomoxef                       5196.0
Cefazolin                      2371.0
Cefixime                       2166.0
Ciprofloxacin                  2071.0
Azithromycin                   2053.0
Cefuroxime                     1691.0
Piperacillin/Tazobactam        1511.0
Cefoperazone/sulbactam         1412.0
Peramivir                      1119.0
Baloxavir marboxil             1085.0
Metronidazole                  1020.0
Cefadroxil                      918.0
Oseltamivir                     835.0
Levofloxacin                    720.0
Clindamycin                     624.0
Gentamicin                      623.0
Ceftriaxone                     610.0
Cephalexin                      517.0
dtype: float64

In [16]:
df_rank = data_filter.melt(
    id_vars=base_cols,
    value_vars=final_cols, # select_drug
    var_name='drug',
    value_name='label'
)

df_rank['label'] = df_rank['label'].fillna(0).astype(int)

In [17]:
df_rank

,ACCOUNTNO,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag,drug,label
0,I11300000002,1,0,0,0,0,0,0,Amoxicillin/Clavulanic acid,0
1,I11300000003,0,0,0,0,0,0,0,Amoxicillin/Clavulanic acid,1
2,I11300000008,0,0,0,0,0,0,0,Amoxicillin/Clavulanic acid,0
3,I11300000011,0,0,0,0,0,0,0,Amoxicillin/Clavulanic acid,0
4,I11300000014,0,0,0,0,0,0,0,Amoxicillin/Clavulanic acid,0
...,...,...,...,...,...,...,...,...,...,...
531387,I11400060686,1,0,0,0,0,0,0,Piperacillin/Tazobactam,0
531388,I11400060687,0,1,0,0,0,0,0,Piperacillin/Tazobactam,1
531389,I11400060701,0,1,0,0,0,0,0,Piperacillin/Tazobactam,0
531390,I11400060720,1,0,0,0,0,0,0,Piperacillin/Tazobactam,1


In [18]:
drug_to_group = dict(zip(table14_last['drug'], table14_last['AUTIBIOTIC_GROUP']))
df_rank['AUTIBIOTIC_GROUP'] = df_rank['drug'].map(drug_to_group)

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_rank['drug_id'] = le.fit_transform(df_rank['drug'])
df_rank = df_rank.drop(columns='drug')

In [20]:
file_path1 = r'D:\ML Data\Spesis\table1_clear.csv'
df1 = pd.read_csv(file_path1, encoding='big5')
file_path2 = r'D:\ML Data\Spesis\table2_clear.csv'
df2 = pd.read_csv(file_path2, encoding='big5')
file_path5 = r'D:\ML Data\Spesis\table5_clear.csv'
df5 = pd.read_csv(file_path5, encoding='big5')
file_path6= r'D:\ML Data\Spesis\table6_clear.csv'
df6 = pd.read_csv(file_path6, encoding='big5')
file_path10 = r'D:\ML Data\Spesis\table10_clear.csv'
df10 = pd.read_csv(file_path10, encoding='big5')

In [21]:
data = pd.merge(df_rank, df1, on='ACCOUNTNO', how='left')
print(data.shape)
data = pd.merge(data, df2, on='ACCOUNTNO', how='left')
print(data.shape)
data = pd.merge(data, df10, on='ACCOUNTNO', how='left')
print(data.shape)
data = pd.merge(data, df6, on='ACCOUNTNO', how='left')
print(data.shape)
data = pd.merge(data, df5, on='ACCOUNTNO', how='left')
print(data.shape)

(531392, 16)
(531392, 26)
(531392, 34)
(531392, 63)
(531392, 71)


In [22]:
roomno_mapping = {'A': '1', 'C': '2', 'D': '3', 'E': '4', 'H': '5', 'K': '6'}
data['ROOMNO'] = data['ROOMNO'].map(roomno_mapping)          

data['SEX'] = data['SEX'].map({'M': 1, 'F': 0})

yn_cols = [
    'ISSEPSIS0', 'FEVER', 'DM', 'CARDIOVASCULAR', 
    'RESPIRATORY', 'CNS', 'CANCER', 'LIVER', 'KIDNEY', 'AUTOIMMUNE'
]

for col in yn_cols:
    data[col] = data[col].map({'Y': 1, 'N': 0})

In [23]:
data.columns

Index(['ACCOUNTNO', 'INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3',
       'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9',
       'OTHERINFECTIONSITE_flag', 'label', 'AUTIBIOTIC_GROUP', 'drug_id',
       'ROOMNO', 'AGE', 'SEX', 'StayTime_hours', 'INTIME', 'ISSEPSIS0',
       'VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2',
       'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP', 'FEVER', 'INJURELEVEL', 'DM',
       'CARDIOVASCULAR', 'RESPIRATORY', 'CNS', 'CANCER', 'LIVER', 'KIDNEY',
       'AUTOIMMUNE', 'Hb', 'WBC', 'Ht', 'PLT', 'Lymphocyte', 'Neutrophil Seg.',
       'Absolute Neutrophil count', 'Na', 'K', 'Creatinine', 'GPT', 'CRP',
       'HST', 'Leukocyte level', 'Nitrite level', 'Bacteria level',
       'Microscopic RBC level', 'Microscopic WBC level', 'PH',
       'Influenza Virus A level', 'T.Bilirubin', 'PT', 'INR', 'APTT', 'PCO2',
       'HCO3', 'BE(ecf)', 'O2 SAT', 'FIRST_ORDERTIME', 'CHECKITEM28A',
       'CHECKITEM27', 'CHECKITEM27SCORE', 'CHECKITEM28SCO

In [24]:
####################### Missing 70~80% #######################

# 檢驗

data['Leukocyte level_flag'] = (
     data['Leukocyte level'].fillna('').str.strip().ne('').astype(int))

data['Nitrite level_flag'] = (
     data['Nitrite level'].fillna('').str.strip().ne('').astype(int))

data['Bacteria level_flag'] = (
     data['Bacteria level'].fillna('').str.strip().ne('').astype(int))

data['Microscopic RBC level_flag'] = (
     data['Microscopic RBC level'].fillna('').str.strip().ne('').astype(int))

data['Microscopic WBC level_flag'] = (
     data['Microscopic WBC level'].fillna('').str.strip().ne('').astype(int))

data['PH_flag'] = (
     data['Microscopic RBC level'].fillna('').str.strip().ne('').astype(int))



# 計分

data['CHECKITEM29SCORE_flag'] = (
     data['CHECKITEM29SCORE'].fillna('').str.strip().ne('').astype(int))

data['CHECKITEM30SCORE_flag'] = (
     data['CHECKITEM30SCORE'].fillna('').str.strip().ne('').astype(int))


####################### Missing > 80% #######################

# 檢驗

data['HST_flag'] = (
     data['HST'].fillna('').str.strip().ne('').astype(int))

data['T.Bilirubin_flag'] = (
     data['T.Bilirubin'].fillna('').str.strip().ne('').astype(int))

data['PT_flag'] = (
     data['PT'].fillna('').str.strip().ne('').astype(int))

data['HST_flag'] = (
     data['HST'].fillna('').str.strip().ne('').astype(int))

data['INR_flag'] = (
     data['INR'].fillna('').str.strip().ne('').astype(int))

data['APTT_flag'] = (
     data['APTT'].fillna('').str.strip().ne('').astype(int))

data['PCO2_flag'] = (
     data['PCO2'].fillna('').str.strip().ne('').astype(int))

data['HCO3_flag'] = (
     data['HCO3'].fillna('').str.strip().ne('').astype(int))


data['BE(ecf)_flag'] = (
     data['BE(ecf)'].fillna('').str.strip().ne('').astype(int))

data['O2 SAT_flag'] = (
     data['O2 SAT'].fillna('').str.strip().ne('').astype(int))

# 計分

data['CHECKITEM28A_flag'] = (
     data['CHECKITEM28A'].fillna('').str.strip().ne('').astype(int))

data['CHECKITEM27_flag'] = (
     data['CHECKITEM27'].fillna('').str.strip().ne('').astype(int))

data['CHECKITEM27SCORE_flag'] = (
     data['CHECKITEM27SCORE'].fillna('').str.strip().ne('').astype(int))

data['CHECKITEM28SCORE_flag'] = (
     data['CHECKITEM28SCORE'].fillna('').str.strip().ne('').astype(int))

data['CHECKITEM31SCORE_flag'] = (
     data['CHECKITEM31SCORE'].fillna('').str.strip().ne('').astype(int))

data['CHECKITEM32SCORE_flag'] = (
     data['CHECKITEM32SCORE'].fillna('').str.strip().ne('').astype(int))

In [25]:
data = data.drop(columns=['ROOMNO', 'INTIME', 'FIRST_ORDERTIME', 'Influenza Virus A level'])

In [26]:
patients = data['ACCOUNTNO'].unique()

train_p, test_p = train_test_split(patients, test_size=0.15, random_state=123)

df_train = data[data['ACCOUNTNO'].isin(train_p)]
df_test  = data[data['ACCOUNTNO'].isin(test_p)]

print(len(df_train['ACCOUNTNO'].unique()))
print(len(df_test['ACCOUNTNO'].unique()))

# 或者用時間切分（更接近臨床）

# INTIME
# ORDERTIME

# 可以做：

# train_df = df_rank[df_rank['INTIME'] < '2023-01-01']
# test_df  = df_rank[df_rank['INTIME'] >= '2023-01-01']

23772
4196


In [27]:
df_train.isnull().sum(), df_test.isnull().sum()

(ACCOUNTNO                0
 INFECTIONSITE1           0
 INFECTIONSITE2           0
 INFECTIONSITE3           0
 INFECTIONSITE4           0
                         ..
 CHECKITEM27_flag         0
 CHECKITEM27SCORE_flag    0
 CHECKITEM28SCORE_flag    0
 CHECKITEM31SCORE_flag    0
 CHECKITEM32SCORE_flag    0
 Length: 90, dtype: int64,
 ACCOUNTNO                0
 INFECTIONSITE1           0
 INFECTIONSITE2           0
 INFECTIONSITE3           0
 INFECTIONSITE4           0
                         ..
 CHECKITEM27_flag         0
 CHECKITEM27SCORE_flag    0
 CHECKITEM28SCORE_flag    0
 CHECKITEM31SCORE_flag    0
 CHECKITEM32SCORE_flag    0
 Length: 90, dtype: int64)

In [28]:
df_train.dtypes, df_test.dtypes

(ACCOUNTNO                object
 INFECTIONSITE1            int64
 INFECTIONSITE2            int64
 INFECTIONSITE3            int64
 INFECTIONSITE4            int64
                           ...  
 CHECKITEM27_flag          int64
 CHECKITEM27SCORE_flag     int64
 CHECKITEM28SCORE_flag     int64
 CHECKITEM31SCORE_flag     int64
 CHECKITEM32SCORE_flag     int64
 Length: 90, dtype: object,
 ACCOUNTNO                object
 INFECTIONSITE1            int64
 INFECTIONSITE2            int64
 INFECTIONSITE3            int64
 INFECTIONSITE4            int64
                           ...  
 CHECKITEM27_flag          int64
 CHECKITEM27SCORE_flag     int64
 CHECKITEM28SCORE_flag     int64
 CHECKITEM31SCORE_flag     int64
 CHECKITEM32SCORE_flag     int64
 Length: 90, dtype: object)

In [29]:
df_train = df_train.copy()
df_test = df_test.copy()

num_cols = ['AGE', 'StayTime_hours', 'VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP', 'Hb', 'WBC', 
            'Ht', 'PLT', 'Lymphocyte', 'Neutrophil Seg.', 'Absolute Neutrophil count', 'Na', 'K', 'Creatinine', 'GPT', 'CRP']

for col in num_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')

In [30]:
# fill score

score_cols = ['Leukocyte level_flag', 'Nitrite level_flag', 'Bacteria level_flag', 'Microscopic RBC level_flag', 'Microscopic WBC level_flag', 'PH_flag',
              'CHECKITEM29SCORE_flag', 'CHECKITEM30SCORE_flag', # 70~80% missing
              'HST_flag','T.Bilirubin_flag', 'PT_flag', 'HST_flag', 'INR_flag', 'APTT_flag', 'PCO2_flag', 'HCO3_flag', 'BE(ecf)_flag',
              'O2 SAT_flag', 'CHECKITEM28A_flag', 'CHECKITEM27_flag', 'CHECKITEM27SCORE_flag', 'CHECKITEM28SCORE_flag', 
              'CHECKITEM31SCORE_flag', 'CHECKITEM32SCORE_flag'] # > 80% missing

for col in score_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    df_train[col] = df_train[col].fillna(-1)

    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')
    df_test[col] = df_test[col].fillna(-1)

In [31]:
drop_cols = ['HST', 'Leukocyte level' , 'Nitrite level', 'Bacteria level', 'Microscopic RBC level', 'Microscopic WBC level' , 
             'PH', 'T.Bilirubin', 'PT' , 'INR', 'APTT', 'PCO2' , 'HCO3', 'BE(ecf)', 'O2 SAT', 
             'CHECKITEM28A', 'CHECKITEM27', 'CHECKITEM27SCORE', 'CHECKITEM28SCORE', 'CHECKITEM29SCORE', 'CHECKITEM30SCORE', 
             'CHECKITEM31SCORE', 'CHECKITEM32SCORE']

df_train = df_train.drop(columns=drop_cols)
df_test = df_test.drop(columns=drop_cols)

In [32]:
scaled_cols = ['AGE', 'StayTime_hours', 'VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP', 'Hb', 'WBC', 
               'Ht', 'PLT', 'Lymphocyte', 'Neutrophil Seg.', 'Absolute Neutrophil count', 'Na', 'K', 'Creatinine', 'GPT', 'CRP']

scaler = StandardScaler()

df_train[scaled_cols] = scaler.fit_transform(df_train[scaled_cols])
df_test[scaled_cols] = scaler.fit_transform(df_test[scaled_cols])

In [33]:
df_train.columns

Index(['ACCOUNTNO', 'INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3',
       'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9',
       'OTHERINFECTIONSITE_flag', 'label', 'AUTIBIOTIC_GROUP', 'drug_id',
       'AGE', 'SEX', 'StayTime_hours', 'ISSEPSIS0', 'VITALSIGNSBT',
       'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP',
       'VITALSIGNSGCS', 'MAP', 'FEVER', 'INJURELEVEL', 'DM', 'CARDIOVASCULAR',
       'RESPIRATORY', 'CNS', 'CANCER', 'LIVER', 'KIDNEY', 'AUTOIMMUNE', 'Hb',
       'WBC', 'Ht', 'PLT', 'Lymphocyte', 'Neutrophil Seg.',
       'Absolute Neutrophil count', 'Na', 'K', 'Creatinine', 'GPT', 'CRP',
       'Leukocyte level_flag', 'Nitrite level_flag', 'Bacteria level_flag',
       'Microscopic RBC level_flag', 'Microscopic WBC level_flag', 'PH_flag',
       'CHECKITEM29SCORE_flag', 'CHECKITEM30SCORE_flag', 'HST_flag',
       'T.Bilirubin_flag', 'PT_flag', 'INR_flag', 'APTT_flag', 'PCO2_flag',
       'HCO3_flag', 'BE(ecf)_flag', 'O2 SAT_flag', 'CHECKI

In [35]:
# Negative Sampling
# 只保留：
# 正樣本（label=1）
# 少量負樣本（label=0）

def negative_sampling(df, neg_ratio=3):
    sampled = []
    
    for acc, group in df.groupby('ACCOUNTNO'):
        pos = group[group['label'] == 1]
        neg = group[group['label'] == 0]
        
        if len(pos) == 0:
            continue
        
        neg_sampled = neg.sample(
            n=min(len(neg), len(pos) * neg_ratio),
            random_state=123
        )
        
        sampled.append(pd.concat([pos, neg_sampled]))
    
    return pd.concat(sampled)

train_df = negative_sampling(df_train, neg_ratio=3) 
test_df = df_test # test_df 不動

In [36]:
train_df.shape

(108977, 67)

In [37]:
train_df.columns

Index(['ACCOUNTNO', 'INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3',
       'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9',
       'OTHERINFECTIONSITE_flag', 'label', 'AUTIBIOTIC_GROUP', 'drug_id',
       'AGE', 'SEX', 'StayTime_hours', 'ISSEPSIS0', 'VITALSIGNSBT',
       'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP',
       'VITALSIGNSGCS', 'MAP', 'FEVER', 'INJURELEVEL', 'DM', 'CARDIOVASCULAR',
       'RESPIRATORY', 'CNS', 'CANCER', 'LIVER', 'KIDNEY', 'AUTOIMMUNE', 'Hb',
       'WBC', 'Ht', 'PLT', 'Lymphocyte', 'Neutrophil Seg.',
       'Absolute Neutrophil count', 'Na', 'K', 'Creatinine', 'GPT', 'CRP',
       'Leukocyte level_flag', 'Nitrite level_flag', 'Bacteria level_flag',
       'Microscopic RBC level_flag', 'Microscopic WBC level_flag', 'PH_flag',
       'CHECKITEM29SCORE_flag', 'CHECKITEM30SCORE_flag', 'HST_flag',
       'T.Bilirubin_flag', 'PT_flag', 'INR_flag', 'APTT_flag', 'PCO2_flag',
       'HCO3_flag', 'BE(ecf)_flag', 'O2 SAT_flag', 'CHECKI

In [48]:
# 建立 feature
base_features = [col for col in train_df.columns if col not in ['ACCOUNTNO', 'label', 'drug_id', 'AUTIBIOTIC_GROUP']]
features = base_features + ['drug_id', 'AUTIBIOTIC_GROUP']

X_train = train_df[features]
y_train = train_df['label']

X_test = test_df[features]
y_test = test_df['label']

In [49]:
# group（LTR 必備）

train_df = train_df.sort_values('ACCOUNTNO').reset_index(drop=True)
test_df = test_df.sort_values('ACCOUNTNO').reset_index(drop=True)

group_train = train_df.groupby('ACCOUNTNO').size().values
group_test  = test_df.groupby('ACCOUNTNO').size().values

In [50]:
print("group sum:", group_train.sum())
print("X_train len:", len(X_train))

group sum: 108977
X_train len: 108977


In [52]:
model = XGBRanker(
    objective='rank:pairwise',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=123,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    group=group_train
)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=0.8, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          learning_rate=0.05, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=300,
          n_jobs=-1, num_parallel_tree=None, objective='rank:pairwise', ...)

In [53]:
scores = model.predict(X_test)

In [55]:
# reshape 回 matrix
num_drugs = len(final_cols)

scores_matrix = scores.reshape(-1, num_drugs)
y_true_matrix = y_test.values.reshape(-1, num_drugs)

In [56]:
K = 3
topk_idx = np.argsort(scores_matrix, axis=1)[:, -K:]

In [57]:
def recall_at_k(y_true, y_pred_idx, k):
    recalls = []
    for i in range(len(y_true)):
        true_set = set(np.where(y_true[i] == 1)[0])
        pred_set = set(y_pred_idx[i])
        
        if len(true_set) == 0:
            continue
        
        recalls.append(len(true_set & pred_set) / len(true_set))
    
    return np.mean(recalls)

print("Recall@3:", recall_at_k(y_true_matrix, topk_idx, 3))

Recall@3: 0.41656099416687803


In [58]:
print("NDCG@3:", ndcg_score(y_true_matrix, scores_matrix, k=3))

NDCG@3: 0.33279877799942076
